  # [![](https://img.shields.io/badge/Thiết%20kế-stablediffusion.vn-0075ff)](https://stablediffusion.vn) [![](https://img.shields.io/badge/Hướng%20dẫn%20-SD%20Nhập%20Môn-5858e6)](https://stablediffusion.vn/nhap-mon-stable-diffusion/) [![](https://img.shields.io/badge/Thư%20viện-Tài%20nguyên-green)](https://bit.ly/sdvn-lib) [![](https://img.shields.io/badge/Group-Support-0075ff)](https://www.facebook.com/groups/stablediffusion.vn)

*`💡 Hỗ trợ CivitAI, Huggingface với link trực tiếp`*

In [ ]:
# @markdown ### 🌌 **Checkpoint**
Checkpoint_Link = ""  # @param {type:"string"}
Checkpoint_Name = ""  # @param {type:"string"}

# @markdown ### 🧠 **Diffusion Models**
Diffusion_Link = ""  # @param {type:"string"}
Diffusion_Name = ""  # @param {type:"string"}

# @markdown ### 🎀 **Lora**
Lora_Link = ""  # @param {type:"string"}
Lora_Name = ""  # @param {type:"string"}

# @markdown ### 🎮 **Text Encoders**
TextEncoder_Link = ""  # @param {type:"string"}
TextEncoder_Name = ""  # @param {type:"string"}

# @markdown ### 💎 **VAE**
VAE_Link = ""  # @param {type:"string"}
VAE_Name = ""  # @param {type:"string"}


import os, requests, re
from google.colab import drive

# --- KẾT NỐI DRIVE ---
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

# --- CẤU HÌNH ĐƯỜNG DẪN ---
Path = '/content/drive/MyDrive/ComfyUI-Data'
drive_dict = {
    f'{Path}/Model': (Checkpoint_Link, Checkpoint_Name),
    f'{Path}/ComfyModel/diffusion_models': (Diffusion_Link, Diffusion_Name),
    f'{Path}/Lora': (Lora_Link, Lora_Name),
    f'{Path}/ComfyModel/clip': (TextEncoder_Link, TextEncoder_Name),
    f'{Path}/ComfyModel/vae': (VAE_Link, VAE_Name),

}

# --- CÁC HÀM XỬ LÝ HỆ THỐNG ---

def get_unique_path(directory, filename):
    """Kiểm tra và đảm bảo đuôi file hợp lệ (.safetensors), chống trùng file"""
    filename = filename.strip()
    valid_extensions = ('.safetensors', '.pt', '.pth', '.ckpt', '.bin', '.onnx', '.gguf')

    # Kiểm tra nếu tên nhập vào chưa kết thúc bằng một định dạng model hợp lệ
    if not any(filename.lower().endswith(ext) for ext in valid_extensions):
        base = filename
        ext = ".safetensors"
    else:
        base, ext = os.path.splitext(filename)

    counter = 1
    unique_filename = f"{base}{ext}"
    full_path = os.path.join(directory, unique_filename)

    while os.path.exists(full_path):
        unique_filename = f"{base}_{counter}{ext}"
        full_path = os.path.join(directory, unique_filename)
        counter += 1
    return full_path

def process_link(link):
    """Xử lý link HuggingFace và Civitai (giữ nguyên params)"""
    if 'huggingface.co' in link:
        return link.replace('blob', 'resolve')
    if 'civitai.com/models' in link:
        # Tải trang để tìm modelVersionId
        !wget "{link}" -q -O model.html
        with open('model.html', 'r', encoding='utf-8') as f:
            ids = re.findall(r'"modelVersionId":(\d+),', f.read())
            if ids:
                return f'https://civitai.com/api/download/models/{ids[0]}'
    return link

def add_token(link):
    """Thêm token vào link mà không làm hỏng các params hiện có"""
    if any(domain in link for domain in ('civitai.com', 'civitai.red')):
        token = "8c7337ac0c39fe4133ae19a3d65b806f"
        connector = "&" if "?" in link else "?"
        return f"{link}{connector}token={token}"
    return link

# --- QUY TRÌNH TẢI XUỐNG ---

print("🚀 Bắt đầu quá trình kiểm tra và tải xuống...")

for folder, (link, custom_name) in drive_dict.items():
    if not link.strip():
        continue

    # Tạo thư mục nếu chưa có
    if not os.path.exists(folder):
        os.makedirs(folder)

    # Xử lý link và chèn token thông minh
    processed_url = process_link(link)
    final_url = add_token(processed_url)

    if custom_name.strip():
        # Sử dụng tên người dùng nhập + logic chống trùng
        save_path = get_unique_path(folder, custom_name)
        file_name_display = os.path.basename(save_path)
        print(f"📥 Đang tải: {file_name_display} vào {folder.split('/')[-1]}...")
        !wget "{final_url}" -O "{save_path}" -q --show-progress
    else:
        # Tải với tên mặc định từ server
        print(f"📥 Đang tải file mặc định vào {folder.split('/')[-1]}...")
        !wget "{final_url}" -P "{folder}" --content-disposition -q --show-progress

print("\n✅ TẤT CẢ ĐÃ HOÀN TẤT!")

#Custom

In [ ]:
#@title ✨ Custom Download
from google.colab import drive
drive.mount('/content/drive')

!sudo apt-get install -y aria2
Link = "" #@param{type:"string"}
Path = "" #@param{type:"string"}
Name = "" #@param{type:"string"}
!aria2c --console-log-level=error --summary-interval=10 -c -x 16 -s 16 -k 1M {Link} -d {Path} -o {Name}